# Lesson 7: Filtering

Neurocampus course "Signals of the whole brain"

Daria Kleeva

dkleeva@gmail.com

April 22, 2026


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import (firwin, filtfilt, lfilter, butter, iirnotch,
                           freqz, sosfreqz, sosfiltfilt, welch)
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown, Checkbox
 
plt.rcParams.update({
    'figure.figsize': (14, 4),
    'font.size': 13,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

fs = 500 

## The simplest filter: the moving average

Imagine you have a noisy signal. The oldest trick: replace each point with the average of its N neighbors.

In [ ]:
t = np.arange(0, 2, 1 / fs)
clean = np.sin(2 * np.pi * 5 * t)  # slow 5 Hz oscillation
np.random.seed(42)
noisy = clean + np.random.randn(len(t)) * 0.8

In [ ]:
plt.plot(t, noisy)
plt.show()

In [ ]:
def moving_average(x, N):
    kernel = np.ones(N) / N
    return np.convolve(x, kernel, mode='same')
 
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
 
axes[0].plot(t, noisy, 'k', linewidth=0.5, alpha=0.5, label='Noisy')
axes[0].plot(t, clean, 'gray', linewidth=2, linestyle='--', label='True signal')
axes[0].set_title('Before: noisy signal')
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('Amplitude')
axes[0].legend(fontsize=10)
 
smoothed = moving_average(noisy, N=20)
axes[1].plot(t, smoothed, 'steelblue', linewidth=2, label='Averaged (N=20)')
axes[1].plot(t, clean, 'gray', linewidth=2, linestyle='--', label='True signal')
axes[1].set_title('After: averaged over 20 samples')
axes[1].set_xlabel('Time (s)')
axes[1].legend(fontsize=10)
 
plt.tight_layout()
plt.show()

The noise is mostly gone and the slow oscillation is preserved. That's it — you just built your first filter.
But what exactly happened? Let's look at it from the frequency side.

We know how to compute power spectra (Welch, from last time). Let's compare the spectrum before and after averaging.

In [ ]:
f_noisy, psd_noisy = welch(noisy, fs=fs, nperseg=fs)
f_smooth, psd_smooth = welch(smoothed, fs=fs, nperseg=fs)
 
fig, ax = plt.subplots(figsize=(12, 4))
ax.semilogy(f_noisy, psd_noisy, 'k', linewidth=1, alpha=0.5, label='Before (noisy)')
ax.semilogy(f_smooth, psd_smooth, 'steelblue', linewidth=2, label='After (averaged)')
ax.axvline(5, color='green', linestyle='--', alpha=0.4, label='5 Hz (our signal)')
ax.set_title('Averaging suppresses high frequencies → it IS a lowpass filter')
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Power (log)')
ax.set_xlim(0, 60)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

High frequencies are suppressed, low frequencies are preserved. A moving average is a lowpass filter.
Now let's see how the window size N affects what gets through.

In [ ]:
def demo_averaging(N=10):
    smoothed_n = moving_average(noisy, N)
    f_s, psd_s = welch(smoothed_n, fs=fs, nperseg=fs)
 
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
 
    axes[0].plot(t, noisy, 'k', linewidth=0.5, alpha=0.3)
    axes[0].plot(t, smoothed_n, 'steelblue', linewidth=2)
    axes[0].plot(t, clean, 'gray', linewidth=2, linestyle='--', alpha=0.5)
    axes[0].set_title(f'Moving average, N = {N} samples ({N/fs*1000:.0f} ms)')
    axes[0].set_xlabel('Time (s)')
    axes[0].set_ylabel('Amplitude')
 
    axes[1].semilogy(f_noisy, psd_noisy, 'k', linewidth=0.8, alpha=0.3, label='Noisy')
    axes[1].semilogy(f_s, psd_s, 'steelblue', linewidth=2, label='Averaged')
    axes[1].set_title('Spectrum')
    axes[1].set_xlabel('Frequency (Hz)')
    axes[1].set_xlim(0, 60)
    axes[1].legend(fontsize=10)
 
    plt.tight_layout()
    plt.show()
 
interact(
    demo_averaging,
    N=IntSlider(min=2, max=80, step=1, value=10, description='N samples'),
);

We see that larger N leads to more aggressive lowpass filtering.

Now let's look at what the moving average actually does, step by step. For each output sample, it takes N input samples and multiplies each by a **weight**. For a simple moving average, all weights are equal: [1/N, 1/N, ..., 1/N].

But what if we use different weights?

In [ ]:
N = 101  # odd, so it's symmetric
 
# Three different weight vectors
weights_uniform = np.ones(N) / N
weights_triangle = np.bartlett(N); weights_triangle /= weights_triangle.sum()
weights_hann = np.hanning(N); weights_hann /= weights_hann.sum()
 
all_weights = [
    ('Uniform (moving average)', weights_uniform, 'steelblue'),
    ('Triangle', weights_triangle, 'darkorange'),
    ('Hann (bell-shaped)', weights_hann, 'crimson'),
]
 
fig, axes = plt.subplots(len(all_weights), 2, figsize=(14, 3.5 * len(all_weights)))
 
for i, (name, w, color) in enumerate(all_weights):
    axes[i, 0].stem(np.arange(N), w, linefmt=f'{color}', markerfmt='o',
                     basefmt='gray')
    axes[i, 0].set_title(f'{name} weights')
    axes[i, 0].set_xlabel('Sample index')
    axes[i, 0].set_ylabel('Weight')
    axes[i, 0].set_ylim(0, max(w) * 1.3)
 
    filtered = np.convolve(noisy, w, mode='same')
    axes[i, 1].plot(t, noisy, 'k', linewidth=0.5, alpha=0.3)
    axes[i, 1].plot(t, filtered, color=color, linewidth=2)
    axes[i, 1].plot(t, clean, 'gray', linewidth=2, linestyle='--', alpha=0.5)
    axes[i, 1].set_title(f'Filtered with {name.lower()} weights')
    axes[i, 1].set_xlabel('Time (s)')
    axes[i, 1].set_ylabel('Amplitude')
 
plt.suptitle('Different weights → different filtering behavior',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

**Key idea:** the weight vector completely defines the filter.  This weight vector has a special name: the **impulse response**.

## The impulse response - a filter's fingerprint

If you feed a single spike (an impulse) through the filter, what comes out is exactly the weight vector. This makes intuitive sense: when the filter slides over a spike, each weight gets multiplied by 1 (at the spike) or 0 (everywhere else). So the output IS the weights.

In [ ]:
spike = np.zeros(200)
spike[100] = 1.0  
 
# Use the Hann weights as our filter
h = np.convolve(spike, weights_hann, mode='same')
 

h_center = h[100 - N//2 : 100 + N//2 + 1]  
 
fig, axes = plt.subplots(1, 3, figsize=(16, 3.5))
 

axes[0].stem(np.arange(len(spike)), spike, linefmt='k-', markerfmt='ko',
             basefmt='gray')
axes[0].set_title('Input: a single spike')
axes[0].set_xlabel('Sample')
axes[0].set_xlim(85, 115)
axes[0].set_ylabel('Amplitude')
 

axes[1].stem(np.arange(N), h_center, linefmt='crimson', markerfmt='ro',
             basefmt='gray')
axes[1].set_title('Output: the impulse response h[n]')
axes[1].set_xlabel('Sample (relative)')
 

axes[2].stem(np.arange(N), weights_hann, linefmt='crimson', markerfmt='ro',
             basefmt='gray')
axes[2].set_title('Original weights — same thing!')
axes[2].set_xlabel('Sample')
 
plt.suptitle('Impulse in → weights out. The impulse response IS the filter.',
             fontsize=14, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

The output (middle) has exactly the same shape as the original weights (right). If you know the impulse response, you know everything about the filter.

## How is impulse response related to frequency response?

We have the filter's "recipe" in the time domain (the impulse response). But we want to know which **frequencies** it keeps and which it removes.

Remember from last time: the Fourier Transform converts a signal from the time domain to the frequency domain.

So let's just take the FFT of the impulse response. What we get is the **frequency response** — how much gain the filter applies at each frequency.

In [ ]:
fig, axes = plt.subplots(len(all_weights), 2, figsize=(14, 3.5 * len(all_weights)))
 
for i, (name, w, color) in enumerate(all_weights):
    axes[i, 0].stem(np.arange(len(w)), w, linefmt=f'{color}', markerfmt='o',
                     basefmt='gray')
    axes[i, 0].set_title(f'h[n]: {name}')
    axes[i, 0].set_xlabel('Sample')
    axes[i, 0].set_ylabel('Weight')
 
    H = np.fft.rfft(w, n=2048)
    f_H = np.fft.rfftfreq(2048, 1 / fs)
    gain_dB = 20 * np.log10(np.abs(H) + 1e-10)
 
    axes[i, 1].plot(f_H, gain_dB, color=color, linewidth=2)
    axes[i, 1].set_title(f'H(f): frequency response = FFT of h[n]')
    axes[i, 1].set_xlabel('Frequency (Hz)')
    axes[i, 1].set_ylabel('Gain (dB)')
    axes[i, 1].set_xlim(0, 80)
    axes[i, 1].set_ylim(-60, 5)
    axes[i, 1].axhline(-3, color='gray', linestyle=':', alpha=0.4)
 
plt.suptitle('h[n] ←FFT→ H(f):  time domain ↔ frequency domain',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


This is THE most important relationship in all of filtering:

`impulse response h[n]  ←→  frequency response H(f) = FFT(h[n])`

Look at the uniform weights (top right): the frequency response has ugly ripples (sidelobes). This means some high frequencies "leak through."  The Hann weights (bottom right): much cleaner — the sidelobes are much smaller. This is why `firwin` uses shaped windows (like Hann) instead of uniform weights.

## Convolution

We've been using `np.convolve` without explaining it. Let's unpack it.

For each output sample y[n], the filter computes:

 `y[n] = h[0]·x[n] + h[1]·x[n-1] + h[2]·x[n-2] + ... + h[M]·x[n-M]`

 It's a weighted sum of the M most recent input samples,  where the weights are the impulse response coefficients. This operation is called **convolution**.

For the moving average: all weights = 1/N, so it's just a plain average. For shaped filters: the weights emphasize nearby samples and downweight distant ones.

Let's vizualize one step of convolution.

In [ ]:
def plot_convolution_step(x, h, center, axes_row, color='crimson', label=''):

    M = len(h)
    half = M // 2
    idx = np.arange(center - half, center + half + 1)
 
    axes_row[0].plot(x, 'k', linewidth=1.5)
    axes_row[0].axvspan(center - half, center + half, alpha=0.15, color=color)
    axes_row[0].set_title(f'Signal x[n] — window of {M} samples', fontsize=12)
    axes_row[0].set_ylabel('Amplitude')

    axes_row[1].stem(idx, h, linefmt=color, markerfmt='.', basefmt='gray')
    axes_row[1].set_title(f'Weights h[n] ({M} taps, {label})', fontsize=12)
    axes_row[1].set_ylabel('Weight')
    axes_row[1].set_xlim(axes_row[0].get_xlim())
 

    segment = x[center - half : center + half + 1]
    product = segment * h
    output_val = np.sum(product)
 
    axes_row[2].stem(idx, product, linefmt='steelblue', markerfmt='.', basefmt='gray')
    axes_row[2].axhline(output_val, color='green', linewidth=2, linestyle='--',
                        label=f'Sum = {output_val:.3f} → y[{center}]')
    axes_row[2].set_title(f'x × h, then sum → y[{center}] = {output_val:.3f}', fontsize=12)
    axes_row[2].set_xlabel('Sample')
    axes_row[2].set_ylabel('x · h')
    axes_row[2].set_xlim(axes_row[0].get_xlim())
    axes_row[2].legend(fontsize=10)
 
 

h_short = np.hanning(21);  h_short /= h_short.sum()
h_long  = np.hanning(101); h_long  /= h_long.sum()
 

x_conv = noisy[:200]
center = 100  
 
fig, axes = plt.subplots(3, 2, figsize=(16, 8))
 
plot_convolution_step(x_conv, h_short, center, axes[:, 0],
                      color='crimson', label='narrow window')
plot_convolution_step(x_conv, h_long, center, axes[:, 1],
                      color='darkorange', label='wide window')
 
plt.suptitle('Convolution = sliding weighted sum. More weights → smoother output.',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
 

 The filter slides this window across the entire signal, computing a weighted sum at each position.

**Summary:**
- Averaging = lowpass filter
- The weight vector = impulse response h[n]
- FFT of h[n] = frequency response H(f) (what frequencies pass through)
- Filtering = convolution = sliding weighted sum
- Longer filter = wider window = more smoothing (same tradeoff as everywhere)

## From moving average to FIR filter

The moving average works, but its frequency response is messy (remember the ripples). Real filter design tools like `scipy.signal.firwin` create better weight vectors, designed to have a clean frequency response with a sharp cutoff and minimal ripples.

In [ ]:
t = np.arange(0, 4, 1 / fs)
theta = 0.8 * np.sin(2 * np.pi * 5 * t)
alpha = 1.5 * np.sin(2 * np.pi * 10 * t)
beta  = 0.5 * np.sin(2 * np.pi * 22 * t)
line  = 0.6 * np.sin(2 * np.pi * 50 * t)
np.random.seed(42)
noise = np.random.randn(len(t)) * 0.3
test_signal = theta + alpha + beta + line + noise
plt.plot(t, test_signal)

In [ ]:
# Moving average 
N_ma = 100
ma_kernel = np.ones(N_ma) / N_ma
 
# firwin lowpass at 20 Hz, same length
fir_kernel = firwin(N_ma, 20, fs=fs)


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 7))
 
# Impulse responses
axes[0, 0].stem(np.arange(N_ma), ma_kernel, linefmt='darkorange', markerfmt='o',
                basefmt='gray')
axes[0, 0].set_title('Moving average weights')
axes[0, 0].set_ylabel('Weight')
 
axes[0, 1].stem(np.arange(N_ma), fir_kernel, linefmt='steelblue', markerfmt='o',
                basefmt='gray')
axes[0, 1].set_title('firwin weights (lowpass 20 Hz)')
 
# Frequency responses
H_ma = np.fft.rfft(ma_kernel, n=4096)
H_fir = np.fft.rfft(fir_kernel, n=4096)
f_resp = np.fft.rfftfreq(4096, 1 / fs)
 
axes[1, 0].plot(f_resp, 20*np.log10(np.abs(H_ma)+1e-10), 'darkorange', linewidth=2,
                label='Moving average')
axes[1, 0].plot(f_resp, 20*np.log10(np.abs(H_fir)+1e-10), 'steelblue', linewidth=2,
                label='firwin')
axes[1, 0].set_title('Frequency response comparison')
axes[1, 0].set_xlabel('Frequency (Hz)')
axes[1, 0].set_ylabel('Gain (dB)')
axes[1, 0].set_xlim(0, 60)
axes[1, 0].set_ylim(-60, 5)
axes[1, 0].legend(fontsize=10)
 
# Filtered signals
y_ma = np.convolve(test_signal, ma_kernel, mode='same')
y_fir = np.convolve(test_signal, fir_kernel, mode='same')
f_ma, p_ma = welch(y_ma, fs=fs, nperseg=2*fs)
f_fi, p_fi = welch(y_fir, fs=fs, nperseg=2*fs)
f_or, p_or = welch(test_signal, fs=fs, nperseg=2*fs)
 
axes[1, 1].semilogy(f_or, p_or, 'k', linewidth=0.8, alpha=0.3, label='Original')
axes[1, 1].semilogy(f_ma, p_ma, 'darkorange', linewidth=1.5, alpha=0.7, label='MA')
axes[1, 1].semilogy(f_fi, p_fi, 'steelblue', linewidth=2, label='firwin')
axes[1, 1].set_title('Result: firwin gives a cleaner cutoff')
axes[1, 1].set_xlabel('Frequency (Hz)')
axes[1, 1].set_xlim(0, 60)
axes[1, 1].legend(fontsize=10)
 
plt.suptitle('Moving average vs firwin: same idea, better design',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

`firwin` has a much cleaner frequency response. The moving average has ripples that let some high frequencies leak through.

## The filter order

The **order** (number of coefficients) controls how sharp the transition between passband and stopband can be. More weights = sharper cutoff.

In [ ]:
def demo_filter_order(order=51):
    b = firwin(order, 20, fs=fs)
    w, h = freqz(b, 1, worN=4096, fs=fs)
    y = np.convolve(test_signal, b, mode='same')
    f_y, p_y = welch(y, fs=fs, nperseg=2*fs)
 
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
 
    # Impulse response
    axes[0].plot(np.arange(len(b)) / fs * 1000, b, 'steelblue', linewidth=1.5)
    axes[0].set_title(f'Impulse response: {order} coefficients ({order/fs*1000:.0f} ms)')
    axes[0].set_xlabel('Time (ms)')
    axes[0].set_ylabel('Weight')
 
    # Frequency response
    axes[1].plot(w, 20*np.log10(np.abs(h)+1e-10), 'steelblue', linewidth=2)
    axes[1].axhline(-3, color='red', linestyle=':', alpha=0.5)
    axes[1].axvline(20, color='gray', linestyle='--', alpha=0.4)
    axes[1].set_title('Frequency response')
    axes[1].set_xlabel('Frequency (Hz)')
    axes[1].set_ylabel('Gain (dB)')
    axes[1].set_xlim(0, 50)
    axes[1].set_ylim(-60, 5)
 
    # Filtered spectrum
    axes[2].semilogy(f_or, p_or, 'k', linewidth=0.8, alpha=0.3, label='Original')
    axes[2].semilogy(f_y, p_y, 'steelblue', linewidth=2, label='Filtered')
    axes[2].set_title('Result')
    axes[2].set_xlabel('Frequency (Hz)')
    axes[2].set_xlim(0, 60)
    axes[2].legend(fontsize=10)
 
    plt.tight_layout()
    plt.show()
 
interact(
    demo_filter_order,
    order=IntSlider(min=11, max=501, step=10, value=51, description='Order'),
);

Rule of thumb: order = 3 × (fs / transition_bandwidth).

## Linear systems

Everything above only works because filters are **linear systems**.

Linear means: if you filter two signals separately and add the results, you get the same thing as filtering the sum.

`filter(A + B) = filter(A) + filter(B)`

 This is why each frequency passes through the filter independently — the filter cannot mix frequencies together.


In [ ]:
b_lin = firwin(101, 20, fs=fs)
 
sig_A = np.sin(2 * np.pi * 5 * t)
sig_B = np.sin(2 * np.pi * 30 * t)
 

result_1 = lfilter(b_lin, 1, sig_A) + lfilter(b_lin, 1, sig_B)
 

result_2 = lfilter(b_lin, 1, sig_A + sig_B)
 
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
 
axes[0].plot(t[:fs], result_1[:fs], 'steelblue', linewidth=2, label='filter(A) + filter(B)')
axes[0].plot(t[:fs], result_2[:fs], 'crimson', linewidth=2, linestyle='--',
             alpha=0.7, label='filter(A + B)')
axes[0].set_title('Both methods give identical results')
axes[0].set_xlabel('Time (s)')
axes[0].legend(fontsize=10)
 
diff = result_1 - result_2
axes[1].plot(t, diff, 'k', linewidth=1)
axes[1].set_title(f'Difference: {np.max(np.abs(diff)):.1e} (essentially zero)')
axes[1].set_xlabel('Time (s)')
axes[1].ticklabel_format(style='scientific', axis='y', scilimits=(0,0))
 
plt.suptitle('Linearity: filter(A+B) = filter(A) + filter(B)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

Why this matters: since any signal is a sum of sinusoids (Fourier!), and the filter treats each sinusoid independently, we can fully describe a filter by what it does to each frequency. That's the frequency response.

## The four filter types

Now we have the tools to understand four standard filter types. Each is defined by what its frequency response looks like.

In [ ]:
# Lowpass: keep < 30 Hz
b_lp = firwin(201, 30, fs=fs)
y_lp = filtfilt(b_lp, 1, test_signal)
 
# Highpass: keep > 8 Hz
b_hp = firwin(201, 8, fs=fs, pass_zero=False)
y_hp = filtfilt(b_hp, 1, test_signal)
 
# Bandpass: keep 8–13 Hz (alpha)
b_bp = firwin(201, [8, 13], fs=fs, pass_zero=False)
y_bp = filtfilt(b_bp, 1, test_signal)
 
# Notch: remove 50 Hz
b_notch, a_notch = iirnotch(50, 30, fs=fs)
y_notch = filtfilt(b_notch, a_notch, test_signal)
 
filters = [
    ('Lowpass (< 30 Hz)\nKeep slow, remove fast',       y_lp,    b_lp, [1],     'steelblue'),
    ('Highpass (> 8 Hz)\nRemove drifts, keep rhythms',   y_hp,    b_hp, [1],     'crimson'),
    ('Bandpass (8–13 Hz)\nKeep only alpha',              y_bp,    b_bp, [1],     'green'),
    ('Notch (remove 50 Hz)\nRemove line noise', y_notch, b_notch, a_notch, 'darkorange'),
]
 
fig, axes = plt.subplots(len(filters), 2, figsize=(15, 3.5 * len(filters)))
 
for i, (name, y_filt, b, a, color) in enumerate(filters):
    # Frequency response
    w, h = freqz(b, a, worN=4096, fs=fs)
    axes[i, 0].plot(w, 20*np.log10(np.abs(h)+1e-10), color=color, linewidth=2)
    axes[i, 0].set_title(f'{name}', fontsize=11)
    axes[i, 0].set_ylabel('Gain (dB)')
    axes[i, 0].set_xlim(0, 60)
    axes[i, 0].set_ylim(-60, 5)
    axes[i, 0].axhline(-3, color='gray', linestyle=':', alpha=0.4)
    if i == len(filters) - 1:
        axes[i, 0].set_xlabel('Frequency (Hz)')
 
    # Filtered spectrum
    f_f, p_f = welch(y_filt, fs=fs, nperseg=2*fs)
    axes[i, 1].semilogy(f_or, p_or, 'k', linewidth=0.8, alpha=0.3, label='Original')
    axes[i, 1].semilogy(f_f, p_f, color=color, linewidth=1.5, label='Filtered')
    axes[i, 1].set_xlim(0, 60)
    axes[i, 1].legend(fontsize=9)
    if i == len(filters) - 1:
        axes[i, 1].set_xlabel('Frequency (Hz)')
 
plt.tight_layout()
plt.show()

## FIR vs IIR

So far, all our filters (moving average, firwin) compute each output sample as a weighted sum of input samples only:

 `y[n] = b[0]·x[n] + b[1]·x[n-1] + ... + b[M]·x[n-M]`

This is a **FIR** (Finite Impulse Response) filter. The name makes sense: if you feed it a single spike, the output has exactly M+1 non-zero samples and then stops. It's finite.

But what if the output also depended on its own previous values?

Consider this rule:

 `y[n] = x[n] + 0.9 · y[n-1]`

Each output sample = the current input + 90% of the previous output.
That's feedback: the filter recycles its own output.

Let's feed it a single spike and see what happens.

In [ ]:
impulse = np.zeros(300)
impulse[50] = 1.0
 

alpha_fb = 0.9  # feedback coefficient
y_iir_simple = np.zeros_like(impulse)
 
for n in range(len(impulse)):
    y_iir_simple[n] = impulse[n] + alpha_fb * (y_iir_simple[n-1] if n > 0 else 0)
 

h_fir_simple = np.ones(21) / 21
y_fir_simple = np.convolve(impulse, h_fir_simple, mode='same')
 
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
 
axes[0].stem(np.arange(len(impulse)), impulse, linefmt='k-', markerfmt='ko',
             basefmt='gray')
axes[0].set_title('Input: a single spike')
axes[0].set_xlim(40, 120)
axes[0].set_ylabel('Amplitude')
axes[0].set_xlabel('Sample')
 
axes[1].plot(y_fir_simple, 'steelblue', linewidth=2)
axes[1].set_title('FIR output: responds, then stops\n(no memory of past outputs)')
axes[1].set_xlim(40, 120)
axes[1].set_xlabel('Sample')
 
axes[2].plot(y_iir_simple, 'crimson', linewidth=2)
axes[2].set_title('IIR output: keeps decaying\n(feedback recycles the output)')
axes[2].set_xlim(40, 120)
axes[2].set_xlabel('Sample')
 
plt.suptitle('FIR vs IIR: the spike reveals the difference',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

The FIR response (blue) lasts exactly 21 samples and stops. The IIR response (red) decays exponentially but never fully stops — because each output feeds 90% of itself back into the next step. After the spike, the values are: 1, 0.9, 0.81, 0.729, ... (geometric series with ratio 0.9).

This is why it's called Infinite Impulse Response — the output theoretically never reaches zero.

Feedback seems complicated. Why not just use FIR filters for everything? # Because feedback is efficient. Let's try to make a sharp lowpass filter at 20 Hz and compare how many coefficients each type needs.

In [ ]:
#IIR: Butterworth order 4 — gentle roll-off, only 5+5=10 coefficients
b_iir_demo, a_iir_demo = butter(4, 20, fs=fs)
 
# FIR: let's try different lengths and see which one matches
b_fir_short = firwin(21, 20, fs=fs)    # 21 coefficients
b_fir_med   = firwin(51, 20, fs=fs)    # 51 coefficients
b_fir_long  = firwin(201, 20, fs=fs)   # 201 coefficients
 
w_iir, H_iir = freqz(b_iir_demo, a_iir_demo, worN=4096, fs=fs)
 
fig, ax = plt.subplots(figsize=(12, 5))
 
ax.plot(w_iir, 20*np.log10(np.abs(H_iir)+1e-10), 'crimson', linewidth=2.5,
        label=f'IIR Butterworth order 4 (10 coefficients)')
 
for b_fir, n, ls in [(b_fir_short, 21, '--'), (b_fir_med, 51, '-'),
                       (b_fir_long, 201, ':')]:
    w_f, H_f = freqz(b_fir, [1], worN=4096, fs=fs)
    ax.plot(w_f, 20*np.log10(np.abs(H_f)+1e-10), 'steelblue', linewidth=1.5,
            linestyle=ls, label=f'FIR ({n} coefficients)')
 
ax.set_title('IIR achieves a moderate roll-off with very few coefficients.\n'
             'FIR needs ~51 coefficients to match it.')
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Gain (dB)')
ax.set_xlim(0, 50)
ax.set_ylim(-60, 5)
ax.axvline(20, color='gray', linestyle=':', alpha=0.4)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()
 
print(f'IIR Butterworth 4: {len(b_iir_demo)} b + {len(a_iir_demo)} a = '
      f'{len(b_iir_demo)+len(a_iir_demo)} coefficients total')

for a given transition steepness, IIR needs fewer coefficients than FIR. But FIR can go much sharper if you're willing to pay with more taps. And FIR is always stable and has linear phase — IIR has neither guarantee.

What does `butter(4, 20, fs=500)` mean?
- `4` — filter order (higher = steeper roll-off, but more coefficients and worse phase distortion)
- `20` — cutoff frequency in Hz
- `fs=500` — sampling rate
- It returns `b, a` — two coefficient arrays for the feedforward and feedback parts of the filter equation.

## The general filter equation

Now we can write the general formula that covers all digital filters:

 `y[n] = b[0]·x[n] + b[1]·x[n-1] + ... + b[M]·x[n-M]`
 `       − a[1]·y[n-1] − a[2]·y[n-2] − ... − a[N]·y[n-N]`

 - The **b** terms = feedforward (weighted sum of inputs) — just like FIR.
 - The **a** terms = feedback (weighted sum of past outputs) — the IIR part.

 For a FIR filter: `a = [1]` (no feedback terms). The equation simplifies
 to just the first line — pure convolution, what we've been doing all along.

 This is why `lfilter(b, a, x)` in scipy takes two coefficient arrays.

In [ ]:
x_demo = test_signal[:200]
b_demo = firwin(51, 15, fs=fs)
 
y_convolve = np.convolve(x_demo, b_demo, mode='full')[:len(x_demo)]
y_lfilter  = lfilter(b_demo, [1], x_demo)   # a = [1] → no feedback
 
print(f'np.convolve vs lfilter(b, [1], x) — max difference: '
      f'{np.max(np.abs(y_convolve - y_lfilter)):.1e}')
print('They are the same operation!')

## The transfer function H(z) — compact notation

The general equation above can be written compactly in the **z-domain** (z⁻¹ means "one sample delay"):

 ```
         b[0] + b[1]·z⁻¹ + b[2]·z⁻² + ... + b[M]·z⁻ᴹ       B(z)
 H(z) = ───────────────────────────────────────────────── = ─────
         1   + a[1]·z⁻¹ + a[2]·z⁻² + ... + a[N]·z⁻ᴺ       A(z)
 ```

 You don't need to compute this — scipy does it for you.
 But knowing the structure explains everything:

 - **FIR:** A(z) = 1 → `H(z) = B(z)` → just a polynomial → always stable.
 - **IIR:** A(z) ≠ 1 → `H(z) = B(z)/A(z)` → a ratio → can be unstable
   if the `a` coefficients are badly chosen (the denominator can blow up).

 This is also why there are two ways to evaluate the frequency response:
 - Plug in z = e^(j2πf/fs) → `H(f) = B(f)/A(f)` → this is what `freqz(b, a)` computes.
 

Let's put it all together: same lowpass task, FIR vs IIR, looking at impulse responses, frequency responses, and the filter coefficients.

In [ ]:
b_fir_demo = firwin(51, 20, fs=fs)
 
impulse_long = np.zeros(500)
impulse_long[100] = 1.0
 
h_fir = lfilter(b_fir_demo, [1], impulse_long)
h_iir = lfilter(b_iir_demo, a_iir_demo, impulse_long)
 
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
 

axes[0, 0].plot(np.arange(len(h_fir)) / fs * 1000, h_fir, 'steelblue', linewidth=1.5)
axes[0, 0].set_title(f'FIR impulse response\n({len(b_fir_demo)} coefficients, stops after '
                     f'{len(b_fir_demo)/fs*1000:.0f} ms)')
axes[0, 0].set_xlabel('Time (ms)')
axes[0, 0].set_ylabel('Amplitude')
 
axes[0, 1].plot(np.arange(len(h_iir)) / fs * 1000, h_iir, 'crimson', linewidth=1.5)
axes[0, 1].set_title(f'IIR impulse response (Butterworth order 4)\n'
                     f'({len(b_iir_demo)}+{len(a_iir_demo)} coefficients, decays forever)')
axes[0, 1].set_xlabel('Time (ms)')
 

axes[1, 0].stem(np.arange(len(b_fir_demo)), b_fir_demo, linefmt='steelblue',
                markerfmt='.', basefmt='gray')
axes[1, 0].set_title(f'FIR coefficients b[n] — {len(b_fir_demo)} values, a = [1]')
axes[1, 0].set_xlabel('Coefficient index')
axes[1, 0].set_ylabel('Value')

ax_iir_coeff = axes[1, 1]
x_pos = np.arange(len(b_iir_demo))
x_pos_a = np.arange(len(a_iir_demo)) + len(b_iir_demo) + 1  # offset for visibility
ax_iir_coeff.stem(x_pos, b_iir_demo, linefmt='crimson', markerfmt='o',
                  basefmt='gray', label=f'b ({len(b_iir_demo)} values)')
ax_iir_coeff.stem(x_pos_a, a_iir_demo, linefmt='darkorange', markerfmt='s',
                  basefmt='gray', label=f'a ({len(a_iir_demo)} values)')
ax_iir_coeff.set_title(f'IIR coefficients — b (feedforward) + a (feedback)')
ax_iir_coeff.set_xlabel('Coefficient index')
ax_iir_coeff.legend(fontsize=10)
 
plt.suptitle('FIR vs IIR: same 20 Hz lowpass, different implementations',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

**Summary — FIR vs IIR:**

 | | FIR | IIR |
 |---|---|---|
 | **Equation** | y[n] = Σ b[k]·x[n-k] | y[n] = Σ b[k]·x[n-k] − Σ a[k]·y[n-k] |
 | **Feedback?** | No (a = [1]) | Yes (a has multiple terms) |
 | **Impulse response** | Finite (stops) | Infinite (decays) |
 | **Transfer function** | H(z) = B(z) | H(z) = B(z) / A(z) |
 | **In scipy** | `b = firwin(...)` | `b, a = butter(...)` |
 | **Efficiency** | Needs many coefficients | Sharp cutoff with few |
 | **Stability** | Always stable | Can be unstable |
 | **Phase** | Linear (preserves waveform) | Non-linear |
 | **For EEG** | Safe default for most tasks | Notch filters, real-time |
 

## What can go wrong

### Pitfall 1: filter ringing at sharp transients

Remember Gibbs phenomenon from the previous lesson? Filters face the same problem. A sharp edge forces the filter to "ring" — producing oscillations that don't exist in the original signal.

This is especially dangerous with narrow bandpass filters: the ringing looks like an oscillatory burst.

In [ ]:
t_spike = np.arange(0, 4, 1 / fs)
spike_signal = np.zeros_like(t_spike)
spike_signal[len(t_spike) // 2] = 10
 
np.random.seed(0)
spike_signal += np.random.randn(len(t_spike)) * 0.05
 

b_ring = firwin(301, [8, 13], fs=fs, pass_zero=False)
spike_filtered = filtfilt(b_ring, 1, spike_signal)
 
fig, axes = plt.subplots(2, 1, figsize=(14, 5), sharex=True)
 
axes[0].plot(t_spike, spike_signal, 'k', linewidth=1.5)
axes[0].set_title('Original: a single sharp spike (NO oscillation here!)', fontsize=13)
axes[0].set_ylabel('Amplitude')
axes[0].set_ylim(-1.5, 11)
 
axes[1].plot(t_spike, spike_filtered, 'crimson', linewidth=1.5)
axes[1].set_title('After bandpass 8–13 Hz: a FAKE alpha burst appears!', fontsize=13,
                  color='red')
axes[1].set_ylabel('Amplitude')
axes[1].set_xlabel('Time (s)')
 
plt.tight_layout()
plt.show()

Always compare filtered and unfiltered data. If you see an oscillatory burst only after filtering — be suspicious.

### Pitfall 2: Causal vs. zero-phase filtering

There are two ways to apply a filter:

- **Causal** (`lfilter`): processes the signal forward only. The output at time t uses only past and present inputs. Introduces a time delay.

- **Zero-phase** (`filtfilt`): applies the filter forward, then backward. The two passes cancel the phase distortion, so there is no delay. But it uses future data — so it's only possible offline (not in real-time).

In [ ]:
onset_signal = np.zeros_like(t)
onset_signal[fs:] = np.sin(2 * np.pi * 10 * t[fs:])  # starts at t = 1.0 s
 
b_onset = firwin(201, 15, fs=fs)
 
y_causal = lfilter(b_onset, 1, onset_signal)
y_zerophase = filtfilt(b_onset, 1, onset_signal)
 
delay_samples = (len(b_onset) - 1) // 2
delay_s = delay_samples / fs
 
fig, axes = plt.subplots(3, 1, figsize=(14, 7), sharex=True)
 
axes[0].plot(t, onset_signal, 'k', linewidth=1.5)
axes[0].axvline(1.0, color='red', linestyle='--', alpha=0.5)
axes[0].set_title('Original: oscillation starts at t = 1.0 s')
axes[0].set_ylabel('Amplitude')
 
axes[1].plot(t, y_zerophase, 'steelblue', linewidth=1.5)
axes[1].axvline(1.0, color='red', linestyle='--', alpha=0.5, label='True onset')
axes[1].set_title('filtfilt (zero-phase): timing is preserved — use for offline analysis')
axes[1].set_ylabel('Amplitude')
axes[1].legend(fontsize=10)
 
axes[2].plot(t, y_causal, 'crimson', linewidth=1.5)
axes[2].axvline(1.0, color='red', linestyle='--', alpha=0.5, label='True onset')
axes[2].axvline(1.0 + delay_s, color='blue', linestyle='--', alpha=0.7,
                label=f'Apparent onset (+{delay_s*1000:.0f} ms delay)')
axes[2].set_title(f'lfilter (causal): onset delayed by {delay_s*1000:.0f} ms — for real-time / BCI')
axes[2].set_xlabel('Time (s)')
axes[2].set_ylabel('Amplitude')
axes[2].legend(fontsize=10)
 
plt.suptitle('Zero-phase vs causal: timing matters',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
 

### Pitfall 3: the sharpness-ringing tradeoff

A sharper cutoff (higher order) means more ringing.

In [ ]:
step = np.zeros(4000)
step[2000:] = 1.0
 
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
 
for order, color in [(51, 'green'), (201, 'steelblue'), (801, 'crimson')]:
    b = firwin(order, 13, fs=fs)
 
    w, h = freqz(b, 1, worN=4096, fs=fs)
    axes[0].plot(w, 20*np.log10(np.abs(h)+1e-10), color=color, linewidth=2,
                 label=f'Order {order}')
 
    y = filtfilt(b, 1, step)
    axes[1].plot(np.arange(len(step)) / fs, y, color=color, linewidth=1.5,
                 label=f'Order {order}')
 
axes[0].set_title('Frequency response')
axes[0].set_xlabel('Frequency (Hz)')
axes[0].set_ylabel('Gain (dB)')
axes[0].set_xlim(0, 30)
axes[0].set_ylim(-60, 5)
axes[0].legend(fontsize=10)
 
axes[1].set_title('Step response — sharper filter → worse ringing')
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Amplitude')
axes[1].set_xlim(3.5, 4.5)
axes[1].legend(fontsize=10)
 
plt.tight_layout()
plt.show()

Green (low order): gentle transition, no ringing, but frequencies above 13 Hz leak through.

Red (high order): sharp cutoff, but terrible ringing around the step. Don't over-engineer your filter.

### Pitfall 4: Highpass filters destroy slow ERPs

In [ ]:
t_erp = np.arange(0, 2.0, 1 / fs)
p300 = 5.0 * np.exp(-0.5 * ((t_erp - 0.3) / 0.05) ** 2)
np.random.seed(7)
erp_signal = p300 + 0.5 * np.sin(2 * np.pi * 10 * t_erp) + np.random.randn(len(t_erp)) * 0.3
 
fig, axes = plt.subplots(2, 2, figsize=(14, 7))
 
for ax, cutoff, col in zip(axes.flat, [0.1, 1.0, 2.0, 5.0],
                            ['green', 'steelblue', 'darkorange', 'crimson']):
    b_hp_erp = firwin(201, cutoff, fs=fs, pass_zero=False)
    y_hp_erp = filtfilt(b_hp_erp, 1, erp_signal)
 
    ax.plot(t_erp * 1000, erp_signal, 'k', linewidth=0.8, alpha=0.3, label='Original')
    ax.plot(t_erp * 1000, y_hp_erp, color=col, linewidth=2, label=f'HP = {cutoff} Hz')
    ax.plot(t_erp * 1000, p300, 'k--', linewidth=1, alpha=0.4, label='True P300')
    ax.axvline(300, color='gray', linestyle=':', alpha=0.4)
    ax.set_title(f'Highpass = {cutoff} Hz')
    ax.set_xlabel('Time (ms)')
    ax.set_ylabel('Amplitude')
    ax.legend(fontsize=8, loc='upper right')
    ax.set_xlim(0, 800)
 
plt.suptitle('Higher highpass cutoff → more ERP distortion',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
 

### Pitfall 5: Notch filter width

 A notch filter must be wide enough to catch the line noise but narrow enough to spare nearby brain activity

In [ ]:
t_notch = np.arange(0, 4, 1 / fs)
gamma_48 = 0.4 * np.sin(2 * np.pi * 48 * t_notch)  # real gamma near 50 Hz
line_50  = 1.0 * np.sin(2 * np.pi * 50 * t_notch)
np.random.seed(42)
notch_signal = gamma_48 + line_50 + np.random.randn(len(t_notch)) * 0.2
 
b_n1, a_n1 = iirnotch(50, 50, fs=fs)  # narrow (Q=50)
b_n2, a_n2 = iirnotch(50, 5, fs=fs)   # wide (Q=5)
 
f_o, p_o = welch(notch_signal, fs=fs, nperseg=2*fs)
f_1, p_1 = welch(filtfilt(b_n1, a_n1, notch_signal), fs=fs, nperseg=2*fs)
f_2, p_2 = welch(filtfilt(b_n2, a_n2, notch_signal), fs=fs, nperseg=2*fs)
 
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
 
axes[0].semilogy(f_o, p_o, 'k', linewidth=0.8, alpha=0.4, label='Original')
axes[0].semilogy(f_1, p_1, 'steelblue', linewidth=1.5, label='Narrow (Q=50)')
axes[0].set_title('Narrow notch: 50 Hz gone, 48 Hz gamma preserved ✓')
axes[0].set_xlim(40, 60)
axes[0].legend(fontsize=9)
axes[0].set_xlabel('Frequency (Hz)')
axes[0].set_ylabel('Power')
 
axes[1].semilogy(f_o, p_o, 'k', linewidth=0.8, alpha=0.4, label='Original')
axes[1].semilogy(f_2, p_2, 'crimson', linewidth=1.5, label='Wide (Q=5)')
axes[1].set_title('Wide notch: 50 Hz gone, but 48 Hz gamma ALSO killed ✗')
axes[1].set_xlim(40, 60)
axes[1].legend(fontsize=9)
axes[1].set_xlabel('Frequency (Hz)')
 
plt.suptitle('Notch width: too wide = collateral damage to real activity',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


## Refreshing spectral analysis and filtering with MNE Python

In [ ]:
import mne


raw = mne.io.read_raw_edf('/Users/dkleeva/Library/CloudStorage/GoogleDrive-dkleeva@gmail.com/My Drive/Teaching/Сигналы целого мозга 2026/Scripts/Data/EEG/probes.edf',
preload=True)

montage = mne.channels.make_standard_montage('standard_1020')
raw.set_montage(montage)

raw.crop(314, 396)



In [ ]:
psd = raw.compute_psd(method='welch',fmax=40)
psd.plot()
plt.show()

In [ ]:
psd = raw.compute_psd(method='multitaper',fmax=40)
psd.plot()
plt.show()

In [ ]:
psd = raw.compute_psd(method='multitaper')
psd.plot()
plt.show()

In [ ]:
raw.info['sfreq']

In [ ]:
raw.resample(250)

In [ ]:
psd = raw.compute_psd(method='multitaper')
psd.plot()
plt.show()

In [ ]:
%matplotlib qt
raw.plot()

In [ ]:
raw_filt = raw.copy().filter(8,12)

In [ ]:
raw_filt.plot()

In [ ]:
raw_filt.plot_psd()

In [ ]:
raw_filt = raw.copy().filter(8,12, method='iir')
raw_filt.plot()

In [ ]:
raw_filt.plot_psd()